In [ ]:
# ====================================================================
# PRODUCTION PIPELINE: High-Speed Vectorized row-blending (h-blend_hs)
# Co-developed by Nina & Google AI Asset Management Group
# Designed for Kaggle Playground Series - Season 6 Episode 9
# ====================================================================

import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

def h_blend_hs(params, df_sample, target_col='Will_Buy_EV'):    
    """
    Executes high-speed matrix vectorized blending on pre-ranked files.
    Preserves the precise non-linear step gaps of the combined rank spaces.
    """
    # 1. Read files directly as raw values since inputs are already rank-prepped
    subms_datas = []
    for subm in params['subm']:
        file_path = os.path.join(params['path'], f"{subm['name']}.csv")
        if os.path.exists(file_path):
            pred_series = pd.read_csv(file_path)['Will_Buy_EV'].values
        else:
            raise FileNotFoundError(f"Critical base submission file missing: {file_path}")
        subms_datas.append(pred_series)
        
    # Stack array for high-velocity row operations
    matrix = np.vstack(subms_datas).T
    n_rows, n_cols = matrix.shape
    
    main_wts = np.asarray([subm['weight'] for subm in params['subm']], dtype=float)
    correct_wts = np.asarray(params['subwts'], dtype=float)
    
    # 2. Vectorized sorting and weight injection
    def compute_space_trend(reverse_sort=False):
        # High-speed row-wise argsort using NumPy matrix operations
        order = np.argsort(-matrix if reverse_sort else matrix, axis=1)
        accumulated_out = np.zeros(n_rows, dtype=float)
        
        for rank_idx in range(n_cols):
            col_idx = order[:, rank_idx]
            # Dynamic rank-power weight reallocation
            dynamic_weights = main_wts[col_idx] + correct_wts[rank_idx]
            accumulated_out += matrix[np.arange(n_rows), col_idx] * dynamic_weights
        return accumulated_out

    # 3. Process directional target trajectories
    asc_trend = compute_space_trend(reverse_sort=False)
    desc_trend = compute_space_trend(reverse_sort=True)

    # 4. Synthesize final prediction matrix on optimal 30x70 blend basis
    # Keeping raw combined values intact without final re-ranking
    df_sample[target_col] = asc_trend * params['type_sort'][1] + desc_trend * params['type_sort'][2]
    return df_sample

# ====================================================================
# MAIN EXECUTION DECK
# ====================================================================
if __name__ == '__main__':
    print("Launching Stratospheric High-Speed Blend Core (No Re-ranking)...")
    
    # Path setup referencing the dataset v11
    input_path = '/kaggle/input/datasets/nina2025/ps-s6e9-11/'
    sample_sub = '/kaggle/input/competitions/playground-series-s6e9/sample_submission.csv'
    
    if not os.path.exists(input_path):
        input_path = './'
    if not os.path.exists(sample_sub):
        df_template = pd.DataFrame({'id': np.arange(668665, 668670)})
    else:
        df_template = pd.read_csv(sample_sub)

    # CONFIGURATION MATRIX: 95% Core with Non-100% Asymmetric Sub-weights
    params_matrix = {
        'path'     : input_path,
        'id_target': ['id', 'Will_Buy_EV'],
        'type_sort': ['asc/desc', 0.30, 0.70],
        'subwts'   : [+0.045, -0.010, -0.015, -0.020], # Custom sub-weights adjustments
        'subm'     : [ 
            {'name': '0.94644',   'weight': +0.95},      # Main Leader
            {'name': '0.94643.a', 'weight': +0.03},      # Co-pilot 1
            {'name': '0.94643.b', 'weight': -0.015},     # Co-pilot 2
            {'name': '0.94643.c', 'weight': +0.03}       # Co-pilot 3
        ]
    }
    
    # Execute matrix blend execution without final ranking smoothing
    submission_df = h_blend_hs(params_matrix, df_template)
    
    # Generate and export final pipeline artifact
    submission_df.to_csv('submission.csv', index=False)
    print("Execution complete! submission.csv written to disk with pure trend gaps.")
    print(submission_df.head(15))
